In [0]:
CREATE OR REFRESH STREAMING TABLE silver.ga_event_cleaned
AS
SELECT
  row.dimensions[0] AS country,
  row.dimensions[1] AS countryId,
  CAST(row.dimensions[2] AS DATE) AS dateProcess,
  CAST(row.metrics[0] AS INT) AS activeUsers,
  CAST(row.metrics[1] AS INT) AS screenPageViews,
  CAST(row.metrics[2] AS INT) AS scrolledUsers,
  CAST(row.metrics[3] AS INT) AS sessions,
  CAST(row.metrics[4] AS INT) AS sessionsPerUser,
  CAST(row.metrics[5] AS INT) AS userEngagementDuration,
  processing_time,file_path
FROM STREAM(dataflatform_dev.bronze.ga_event)
LATERAL VIEW explode(
  from_json(
    content.json,
    'STRUCT<headers:STRUCT<dimensions:ARRAY<STRING>,metrics:ARRAY<STRING>>,rows:ARRAY<STRUCT<dimensions:ARRAY<STRING>,metrics:ARRAY<STRING>>>>'
  ).rows
) AS row
;

In [0]:
CREATE OR REFRESH STREAMING TABLE silver.ga_event_upserted
COMMENT "Streaming table containing latest invoice database."
TBLPROPERTIES("table.layer"="silver", "table.type"="transformation as scd-type1");
CREATE FLOW silver_ga_event_upserted_flow
AS AUTO CDC INTO silver.ga_event_upserted
FROM STREAM(silver.ga_event_cleaned)
KEYS (countryId,dateProcess)
SEQUENCE BY processing_time
COLUMNS * EXCEPT (processing_time, file_path)
STORED AS SCD TYPE 1;